In [ ]:
!hostname

# KVQuant Implementation -- 2-bit only -- REAL packed KV cache

Variant of `KVQuant_2bit_Implementation.ipynb` in which the KV cache is
**genuinely stored compressed**, instead of the upstream repo's simulated
(quantize-then-dequantize-back-to-fp16) storage. Same model, same pinned
packages, same seed, same datasets/loaders/prompts/grading, same metric
definitions, same result-CSV schema as the rest of the notebook family.

**What is real here (vs. the simulation notebook):**

- Per token, per layer, the ONLY persistent K/V storage is: 2-bit codes
  packed four-per-byte into `uint8` tensors, fp16 outlier values with
  `uint8` column + `int32` row indices, per-token fp16 nuq codebooks for V,
  and one static per-channel fp16 codebook per layer for K. Full-precision
  K/V exist only transiently inside a single attention step and are freed
  immediately.
- Memory is therefore **MEASURED** -- the real byte size of the tensors
  actually resident -- exactly the same kind of number the H2O notebooks
  report, not a calculated/simulated figure. The outlier-tracking hooks and
  `measure_bytes_from_tracker` from the simulation notebook are gone; there
  is nothing to simulate.

**What still comes from the official KVQuant source (unchanged):** the whole
calibration pipeline (Fisher via `run-fisher.py`, quantizer construction via
`llama_simquant.py --quantize`, nuq2 with 1% sparse outliers), and every
quantization DECISION at runtime: `make_quant_sim` installs the repo's own
`QuantLinearSim` modules on every `k_proj`/`v_proj`, so the quantized values
(centroids / kept-fp16 outliers) are produced by the repo's code with the
repo's calibrated thresholds; outlier identification reuses the repo's
`get_outliers` / `get_outliers_dynamic` with each module's own stored
thresholds. This notebook adds only the storage layer around those
decisions: encoding the repo-produced values into packed 2-bit form
(Keys per-channel pre-RoPE, Values per-token, matching `llama_simquant.py`'s
main-flow layer matching) and decoding them on read. Encoding is exact --
non-outlier values ARE nuq centroids, so they map losslessly to 2-bit
codebook indices -- and a validation cell asserts bit-exact reconstruction
plus logit agreement against the simulation path before any evaluation runs.

**Because storage is real, attention must read from it.** Each layer's
attention forward is replaced with a functionally-equivalent implementation
that appends the new token's quantized K (pre-RoPE) / V to the packed store,
reconstructs the full K/V from the packed store, applies RoPE to keys at
their absolute positions, and computes attention in plain (eager-math)
PyTorch. Two consequences for cross-method comparison, stated up front:

1. **Latency is real but UNOPTIMIZED.** Dequantize-on-read runs as unfused
   PyTorch ops over the whole cache every step (real deployments fuse this
   into custom CUDA kernels), and the attention math is eager-style -- the
   same "eager tax" class as the H2O notebooks, and unlike the sdpa
   simulation notebooks. Latency here answers "what does THIS
   implementation cost", not "what would KVQuant's deployment kernels
   cost". Compare latency only against an eager-attention baseline run.
2. **Accuracy/perplexity should match the simulation notebook** (the model
   consumes identical values either way). Agreement between the two
   notebooks is itself a correctness check -- small differences can arise
   only from fp16 summation-order effects in the eager attention math.

Run cells top to bottom. Needs a GPU runtime.


## Setup

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods -- identical
# to the rest of the notebook family.

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")


In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.float16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache/robust_call are defined here (not in Helper
# Functions, below) because the Fisher calibration cell later in this same
# Setup section calls robust_call for its wikitext-2 prefetch -- they need
# to exist before that point. sync_if_cuda/clear_memory have no such early
# dependency and live in Helper Functions with the rest of the genuinely
# cross-dataset inference machinery.


In [ ]:
# Block 3 - Experiment settings.
# Every shared quantity is byte-for-byte identical to the rest of the
# notebook family, so the methods' evaluations differ only where the
# compression method itself makes them differ. This notebook only ever
# builds/evaluates ONE bit-width (2-bit), with REAL packed storage.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 0
C = 2048                    # chunk length for WikiText-103 = model's max context
N_EVAL_SAMPLES = 64          # caps WikiText-103 source prompts (chunk count)
N_QA_SAMPLES = 256           # GSM8K and ARC-Challenge question count
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "kvquant_2bit_real"

ABITS = 2                   # this notebook's single bit-width
SPARSITY_THRESHOLD = 0.99   # keep the extreme ~1% of values in full precision
FISHER_OUTPUT_DIR = "/content/fisher-llama-3.1-8b-c2048"
QUANTIZER_PATH = "/content/quantizers_llama-3.1-8b_v3_c2048_abits2.pickle"

random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation briefly, then end with exactly this format:\n"
    "#### <final number>\n\n"
    "Question: Mia has 3 marbles and buys 4 more. How many marbles does she have?\n"
    "Answer: Mia has 3 + 4 = 7 marbles.\n"
    "#### 7\n\n"
    "Question: A box has 6 pencils. Sam buys 5 boxes. How many pencils does Sam buy?\n"
    "Answer: Sam buys 6 * 5 = 30 pencils.\n"
    "#### 30\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Chunk length C:", C)
print("Bit-width:", ABITS, "| sparsity threshold:", SPARSITY_THRESHOLD)
print("N_EVAL_SAMPLES (WikiText-103 chunks):", N_EVAL_SAMPLES)
print("N_QA_SAMPLES (GSM8K/ARC-Challenge questions):", N_QA_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)


In [ ]:
# Repo setup 1/3 - Clone repo (always fresh, so the rotary_emb patch below
# always applies to pristine upstream source rather than a possibly-
# already-patched copy left over from an earlier run in this session).
import os, shutil


def check_path(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(f"ERROR: {label} not found at {path}")
    print(f"OK: {label} found at {path}")


if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone --recurse-submodules https://github.com/yoshikodes/KVCacheCompression.git

assert os.path.exists("/content/KVCacheCompression/KVQuant/gradients/run-fisher.py"), \
    "ERROR: run-fisher.py not found! Clone may have failed."
assert os.path.exists("/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"), \
    "ERROR: llama_simquant.py not found! Clone may have failed."
print("Repo verified OK")


In [ ]:
# Repo setup 2/3 - Install gradients (Fisher calibration) dependencies
check_path("/content/KVCacheCompression/KVQuant/gradients", "gradients dir")

%cd /content/KVCacheCompression/KVQuant/gradients
!pip install -e . -q
!pip install datasets==2.14.5 sentencepiece==0.1.99 accelerate -q -U
!pip install peft==0.6.0 -q
print("Gradients deps installed")


In [ ]:
# Patch - llama_simquant.py: move the shared rotary embedding module to the
# target device before calibration. transformers==4.43.4 computes rotary
# embeddings ONCE at the top-level LlamaModel, not per-layer -- the upstream
# script's custom per-layer CPU->GPU transfer (written for an older
# architecture) never moves this shared module, so it stays on CPU while
# everything it's multiplied against is on GPU. Only needed for the
# calibration path here (this notebook's own evaluation, below, uses a
# normally-loaded model and never calls llama_simquant.py's llama_eval).
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    _content = f.read()

if "model.model.rotary_emb = model.model.rotary_emb.to(DEV)" not in _content:
    _m = re.search(r"^([ \t]*)quantizers = llama_calibration\(", _content, re.MULTILINE)
    assert _m, "ERROR: could not locate the llama_calibration(...) call to patch"
    _indent = _m.group(1)
    _insertion = (
        f"{_indent}if hasattr(model, 'model') and hasattr(model.model, 'rotary_emb'):\n"
        f"{_indent}    model.model.rotary_emb = model.model.rotary_emb.to(DEV)\n"
    )
    _content = _content[:_m.start()] + _insertion + _content[_m.start():]
    with open(filepath, "w") as f:
        f.write(_content)
    print("Patched llama_simquant.py: rotary_emb moved to device before calibration")
else:
    print("rotary_emb device patch already applied, skipping")


In [ ]:
# Patch - llama_simquant.py: get_model() hardcodes
# use_flash_attention_2=True, which crashes with "flash_attn seems to be not
# installed" since this project never installs the flash-attn package (it's
# optional elsewhere and skipped here entirely). Replace it with a
# GPU-capability-aware attn_implementation choice that also respects a
# FORCE_ATTN_IMPL env var. Only llama_simquant.py's internal get_model()
# needs patching, for the Quantize step -- this notebook's own in-process
# loader builds its own model_kwargs directly.
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    _content = f.read()

_old_load = "model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, use_flash_attention_2=True, torch_dtype=torch.half)"
_new_load = (
    "import warnings as _warnings\n"
    "    import os as _attn_os\n"
    "    import torch as _torch_gpu_check\n"
    "    _cap = _torch_gpu_check.cuda.get_device_capability(0) if _torch_gpu_check.cuda.is_available() else (0, 0)\n"
    "    _attn_impl = \"flash_attention_2\" if _cap[0] >= 8 else \"sdpa\"\n"
    "    _force_attn = _attn_os.environ.get('FORCE_ATTN_IMPL')\n"
    "    if _force_attn:\n"
    "        _attn_impl = _force_attn\n"
    "    print(f'GPU compute capability: {_cap} -> using attn_implementation={_attn_impl}' + (' (forced)' if _force_attn else ''))\n"
    "    with _warnings.catch_warnings():\n"
    "        _warnings.filterwarnings(\"ignore\", message=\".*Flash Attention 2.0 with a model not initialized on GPU.*\")\n"
    "        model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, attn_implementation=_attn_impl, torch_dtype=torch.half)"
)
if _old_load in _content:
    _content = _content.replace(_old_load, _new_load)
    with open(filepath, "w") as f:
        f.write(_content)
    print("llama_simquant.py attn_implementation patch applied (GPU-capability-aware)")
elif "GPU compute capability" in _content:
    print("attn_implementation patch already applied, skipping")
else:
    print("WARNING: could not find the expected model-loading line to patch.")
print("Has GPU-capability check:", "GPU compute capability" in open(filepath).read())


In [ ]:
# Patch - simquant_module_quantizer.py: QuantLinearSim.__init__ builds
# self.outlier_threshold_upper/lower via torch.tensor(already_a_tensor),
# which PyTorch flags with a UserWarning (cosmetic). Same patch as the
# simulation notebook so the two notebooks run identical repo source.
filepath = "/content/KVCacheCompression/KVQuant/quant/kvquant/simquant_module_quantizer.py"
with open(filepath, "r") as f:
    _content = f.read()

_old_thresh = (
    "        self.outlier_threshold_upper = torch.tensor(quantizer[0]).cuda().flatten().half()\n"
    "        self.outlier_threshold_lower = torch.tensor(quantizer[1]).cuda().flatten().half()"
)
_new_thresh = (
    "        self.outlier_threshold_upper = quantizer[0].clone().detach().cuda().flatten().half()\n"
    "        self.outlier_threshold_lower = quantizer[1].clone().detach().cuda().flatten().half()"
)
if _old_thresh in _content:
    _content = _content.replace(_old_thresh, _new_thresh)
    with open(filepath, "w") as f:
        f.write(_content)
    print("Patched simquant_module_quantizer.py: outlier thresholds built via .clone().detach() instead of torch.tensor(tensor)")
elif "quantizer[0].clone().detach()" in _content:
    print("simquant_module_quantizer.py outlier-threshold patch already applied, skipping")
else:
    print("WARNING: could not find the expected outlier-threshold lines to patch (cosmetic only -- safe to ignore if this shows up).")


In [ ]:
# Patch - gradients/src/transformers: add rope_type="llama3" support to the
# vendored transformers fork run-fisher.py imports (via PYTHONPATH below).
# This fork predates Llama 3.1's NTK-by-parts long-context RoPE scaling --
# its _rope_scaling_validation only accepts the old {"type", "factor"}
# 2-key format, and its rotary embedding classes only implement "linear"/
# "dynamic". Llama-3.1-8B's real config.json rope_scaling is
# {"factor": 8.0, "low_freq_factor": 1.0, "high_freq_factor": 4.0,
# "original_max_position_embeddings": 8192, "rope_type": "llama3"}, which
# the unpatched fork rejects outright (ValueError from AutoConfig, raised
# inside the run-fisher.py subprocess). Patched to match HF's own
# ROPE_INIT_FUNCTIONS["llama3"] formula exactly, so Fisher calibration
# sees the SAME positional encoding the model actually uses -- the
# frequency scaling applies uniformly to every position, not just beyond
# the original 8192 context, so this isn't a cosmetic no-op even at the
# short C=2048 calibration length used here.
config_path = "/content/KVCacheCompression/KVQuant/gradients/src/transformers/models/llama/configuration_llama.py"
modeling_path = "/content/KVCacheCompression/KVQuant/gradients/src/transformers/models/llama/modeling_llama.py"

with open(config_path, "r") as f:
    _cfg_content = f.read()

_old_validation = """        if not isinstance(self.rope_scaling, dict) or len(self.rope_scaling) != 2:
            raise ValueError(
                "`rope_scaling` must be a dictionary with with two fields, `type` and `factor`, "
                f"got {self.rope_scaling}"
            )
        rope_scaling_type = self.rope_scaling.get("type", None)
        rope_scaling_factor = self.rope_scaling.get("factor", None)
        if rope_scaling_type is None or rope_scaling_type not in ["linear", "dynamic"]:
            raise ValueError(
                f"`rope_scaling`'s type field must be one of ['linear', 'dynamic'], got {rope_scaling_type}"
            )
        if rope_scaling_factor is None or not isinstance(rope_scaling_factor, float) or rope_scaling_factor <= 1.0:
            raise ValueError(f"`rope_scaling`'s factor field must be a float > 1, got {rope_scaling_factor}")"""

_new_validation = """        rope_scaling_type = self.rope_scaling.get("type", self.rope_scaling.get("rope_type", None))
        if rope_scaling_type == "llama3":
            required = {"factor", "low_freq_factor", "high_freq_factor", "original_max_position_embeddings"}
            missing = required - set(self.rope_scaling)
            if missing:
                raise ValueError(f"`rope_scaling` of type 'llama3' is missing required field(s): {missing}")
            return
        if not isinstance(self.rope_scaling, dict) or len(self.rope_scaling) != 2:
            raise ValueError(
                "`rope_scaling` must be a dictionary with with two fields, `type` and `factor`, "
                f"got {self.rope_scaling}"
            )
        rope_scaling_factor = self.rope_scaling.get("factor", None)
        if rope_scaling_type is None or rope_scaling_type not in ["linear", "dynamic"]:
            raise ValueError(
                f"`rope_scaling`'s type field must be one of ['linear', 'dynamic'], got {rope_scaling_type}"
            )
        if rope_scaling_factor is None or not isinstance(rope_scaling_factor, float) or rope_scaling_factor <= 1.0:
            raise ValueError(f"`rope_scaling`'s factor field must be a float > 1, got {rope_scaling_factor}")"""

if _old_validation in _cfg_content:
    _cfg_content = _cfg_content.replace(_old_validation, _new_validation)
    with open(config_path, "w") as f:
        f.write(_cfg_content)
    print("Patched configuration_llama.py: rope_scaling validation now accepts rope_type=\'llama3\'")
elif "llama3" in _cfg_content:
    print("configuration_llama.py rope_scaling patch already applied, skipping")
else:
    raise RuntimeError("ERROR: could not locate rope_scaling validation block to patch in configuration_llama.py")

with open(modeling_path, "r") as f:
    _model_content = f.read()

_llama3_class = '''

class LlamaLlama3ScalingRotaryEmbedding(LlamaRotaryEmbedding):
    """LlamaRotaryEmbedding extended with Llama 3\'s NTK-by-parts long-context
    scaling (rope_scaling rope_type="llama3"). Reimplements HF\'s
    ROPE_INIT_FUNCTIONS["llama3"] formula exactly, since this vendored fork
    predates that scaling type."""

    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None,
                 scaling_factor=8.0, low_freq_factor=1.0, high_freq_factor=4.0,
                 original_max_position_embeddings=8192):
        self.scaling_factor = scaling_factor
        self.low_freq_factor = low_freq_factor
        self.high_freq_factor = high_freq_factor
        self.original_max_position_embeddings = original_max_position_embeddings
        super().__init__(dim, max_position_embeddings, base, device)

    def _set_cos_sin_cache(self, seq_len, device, dtype):
        self.max_seq_len_cached = seq_len

        base_inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2, dtype=torch.int64).float().to(device) / self.dim))

        low_freq_wavelen = self.original_max_position_embeddings / self.low_freq_factor
        high_freq_wavelen = self.original_max_position_embeddings / self.high_freq_factor
        wavelen = 2 * math.pi / base_inv_freq

        inv_freq_llama = torch.where(wavelen > low_freq_wavelen, base_inv_freq / self.scaling_factor, base_inv_freq)
        smooth_factor = (
            (self.original_max_position_embeddings / wavelen - self.low_freq_factor)
            / (self.high_freq_factor - self.low_freq_factor)
        )
        smoothed_inv_freq = smooth_factor * inv_freq_llama / self.scaling_factor + (1 - smooth_factor) * inv_freq_llama
        is_medium_freq = ~(wavelen < high_freq_wavelen) * ~(wavelen > low_freq_wavelen)
        inv_freq_llama = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)

        self.register_buffer("inv_freq", inv_freq_llama, persistent=False)

        t = torch.arange(self.max_seq_len_cached, device=device, dtype=torch.int64).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)
'''

_class_anchor = "def rotate_half(x):"
if "class LlamaLlama3ScalingRotaryEmbedding" not in _model_content:
    _idx = _model_content.index(_class_anchor)
    _model_content = _model_content[:_idx] + _llama3_class.lstrip("\n") + "\n\n" + _model_content[_idx:]
    with open(modeling_path, "w") as f:
        f.write(_model_content)
    print("Patched modeling_llama.py: added LlamaLlama3ScalingRotaryEmbedding class")
else:
    print("modeling_llama.py LlamaLlama3ScalingRotaryEmbedding already added, skipping")

with open(modeling_path, "r") as f:
    _model_content = f.read()

_old_init_rope = '''        else:
            scaling_type = self.config.rope_scaling["type"]
            scaling_factor = self.config.rope_scaling["factor"]
            if scaling_type == "linear":'''
_new_init_rope = '''        else:
            scaling_type = self.config.rope_scaling.get("type", self.config.rope_scaling.get("rope_type"))
            scaling_factor = self.config.rope_scaling["factor"]
            if scaling_type == "llama3":
                self.rotary_emb = LlamaLlama3ScalingRotaryEmbedding(
                    self.head_dim,
                    max_position_embeddings=self.max_position_embeddings,
                    scaling_factor=scaling_factor,
                    low_freq_factor=self.config.rope_scaling["low_freq_factor"],
                    high_freq_factor=self.config.rope_scaling["high_freq_factor"],
                    original_max_position_embeddings=self.config.rope_scaling["original_max_position_embeddings"],
                    base=self.rope_theta,
                    device="cpu"
                )
            elif scaling_type == "linear":'''

if _old_init_rope in _model_content:
    _model_content = _model_content.replace(_old_init_rope, _new_init_rope)
    with open(modeling_path, "w") as f:
        f.write(_model_content)
    print("Patched modeling_llama.py: _init_rope now handles rope_type=\'llama3\'")
elif 'scaling_type == "llama3"' in _model_content:
    print("modeling_llama.py _init_rope patch already applied, skipping")
else:
    raise RuntimeError("ERROR: could not locate _init_rope's scaling_type branch to patch in modeling_llama.py")


In [ ]:
# Fisher calibration (gradients) -- seqlen/maxseqlen MUST match the Quantize
# step's seqlen below (C = 2048), or Fisher's saved tensors won't align with
# Quantize's activation tensor shapes elementwise.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pre-fetch wikitext-2 (with retries) in this notebook's own Python process
# before launching the Fisher subprocess -- run-fisher.py internally calls
# load_dataset('wikitext', 'wikitext-2-raw-v1', ...) itself, and a flaky
# connection there can crash the whole subprocess uncaught, leaving
# FISHER_OUTPUT_DIR never created.
print("Pre-fetching wikitext-2 (train+test) with retries...")
robust_call(
    load_dataset, "wikitext", "wikitext-2-raw-v1", split="train",
    desc="wikitext-2 train prefetch", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
)
robust_call(
    load_dataset, "wikitext", "wikitext-2-raw-v1", split="test",
    desc="wikitext-2 test prefetch", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
)
print("wikitext-2 cached.")


# Pre-fetch the Llama-3.1-8B model weights (with retries), using
# huggingface_hub.snapshot_download directly -- NOT transformers'
# AutoModelForCausalLM. By this point in the notebook, the gradients
# install has replaced the real `transformers` package on disk with its
# own vendored fork (intentional -- run-fisher.py's subprocess needs it),
# but this kernel already imported the REAL transformers back in Block 2.
# snapshot_download only touches huggingface_hub, which isn't affected.
from huggingface_hub import snapshot_download

print(f"Pre-fetching {MODEL_ID} weights with retries (this can take a few minutes)...")
robust_call(
    snapshot_download, MODEL_ID,
    desc=f"{MODEL_ID} weights prefetch",
    max_retries=5, backoff_sec=10,
)
print(f"{MODEL_ID} weights cached.")

%cd /content/KVCacheCompression/KVQuant/gradients

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/gradients/src python run-fisher.py --model_name_or_path {MODEL_ID} --output_dir {FISHER_OUTPUT_DIR} --dataset wikitext2 --seqlen {C} --maxseqlen {C} --num_examples 8

check_path(FISHER_OUTPUT_DIR, "Fisher output")
print("Fisher done for Llama-3.1-8B at seqlen", C)


In [ ]:
# Repo setup 3/3 - Install quant (eval-support) dependencies. Runs AFTER
# Fisher calibration since gradients needs tokenizers<0.19 while this pins
# tokenizers==0.19.1 -- installing this first would break Fisher.
check_path("/content/KVCacheCompression/KVQuant/quant", "quant dir")

%cd /content/KVCacheCompression/KVQuant/quant
!pip install -e . -q --no-deps

!pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2"

print("Quant deps installed, package versions pinned to match every other method's notebook")


In [ ]:
# Quantize (nuq2, 1% outliers) -- builds the single quantizer pickle this
# notebook's own in-process eval loop will load later. Note this only runs
# llama_calibration internally (the --quantize branch), never llama_eval --
# so the rotary_emb patch above is the only one needed.
sys.path = [p for p in sys.path if "gradients" not in p]

check_path(FISHER_OUTPUT_DIR, "Fisher output for Llama-3.1-8B")

%cd /content/KVCacheCompression/KVQuant/quant

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/quant FORCE_ATTN_IMPL=sdpa python llama_simquant.py {MODEL_ID} --abits {ABITS} --nsamples 8 --seqlen {C} --nuq --fisher {FISHER_OUTPUT_DIR} --quantize --include_sparse --sparsity-threshold {SPARSITY_THRESHOLD} --quantizer-path {QUANTIZER_PATH}

check_path(QUANTIZER_PATH, "Quantizer pickle for Llama-3.1-8B")
print("Quantization done for Llama-3.1-8B at seqlen", C, "bit-width", ABITS)


In [ ]:
# Block - Load tokenizer + the 2-bit quantized model (via make_quant_sim
# applied in-process, matching llama_simquant.py's own main-flow layer-name
# matching: Keys quantized per-channel pre-RoPE, Values quantized
# per-token). IDENTICAL to the simulation notebook up to this point: the
# repo's QuantLinearSim modules are installed on every k_proj/v_proj and
# remain the sole source of every quantization decision. What changes is
# only what happens AFTER these modules produce their values -- the next
# cells add real packed storage around them instead of letting fp16
# dequantized values persist in the cache.
#
# attn_implementation is set to "sdpa" to match the simulation notebook's
# setting exactly. This has no effect on the packed path's own attention
# math during evaluation -- it replaces each layer's attention forward
# with its own eager-math implementation regardless (it must read from the
# packed store), so the stock kernels are never used once evaluation
# starts. It DOES affect the validation cell's reference pass, which uses
# the stock (unreplaced) forward: that pass now runs through SDPA's fused
# kernel rather than eager math, so its logits can differ very slightly
# more from the packed path's eager-math reconstruction than an eager
# reference would -- the validation cell's argmax-agreement threshold
# already accounts for this class of fp16 summation-order noise.
%cd /content/KVCacheCompression/KVQuant/quant
sys.path.insert(0, "/content/KVCacheCompression/KVQuant/quant")

from kvquant.simquant_module_quantizer import make_quant_sim, get_outliers, get_outliers_dynamic, QuantLinearSim

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "sdpa",
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print(f"Loading model to be quantized at {ABITS}-bit...")
model_q = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_q = model_q.to(DEVICE)
model_q.eval()
model_q.config.use_cache = False   # the packed store below IS the cache

device = next(model_q.parameters()).device

with open(QUANTIZER_PATH, "rb") as f:
    quantizers = pickle.load(f)

PERCHANNEL_MATCH = ["k_proj"]
PERTOKEN_MATCH = ["v_proj"]

perchannelquant = {}
pertokenquant = {}
for _k in quantizers.keys():
    for _p in PERCHANNEL_MATCH:
        if _p in _k:
            perchannelquant[_k] = quantizers[_k]
    for _p in PERTOKEN_MATCH:
        if _p in _k:
            pertokenquant[_k] = quantizers[_k]

print(f"Quantizer covers {len(perchannelquant)} per-channel (K) layers, {len(pertokenquant)} per-token (V) layers")

make_quant_sim(
    model_q, perchannelquant, ABITS,
    perchannel=True, include_sparse=True, sparsity_threshold=SPARSITY_THRESHOLD,
    dynamicquantization=False, nuq=True, nf_nuq=False, norm=False,
    cap_outliers=-1, first_few_fp16=-1, clamp=False,
)
make_quant_sim(
    model_q, pertokenquant, ABITS,
    perchannel=False, include_sparse=True, sparsity_threshold=SPARSITY_THRESHOLD,
    dynamicquantization=True, nuq=True, nf_nuq=False, norm=False,
    cap_outliers=-1, first_few_fp16=-1, clamp=False,
)

_n_qls = sum(1 for _, m in model_q.named_modules() if isinstance(m, QuantLinearSim))
print(f"Quantization applied in place: {_n_qls} QuantLinearSim modules (32 k_proj + 32 v_proj expected).")
print("model_q: nuq", ABITS, "bit, sparsity_threshold", SPARSITY_THRESHOLD, "| storage: REAL packed (next cells)")


In [ ]:
# Block - REAL packed 2-bit KV storage: packing/encoding primitives.
#
# This replaces the simulation notebook's outlier-tracking hooks and
# measure_bytes_from_tracker. Nothing here makes any quantization decision:
# values arrive already quantized by the repo's QuantLinearSim (non-outlier
# values ARE nuq centroids; outliers ARE the kept fp16 values), and outlier
# positions are identified with the repo's own get_outliers /
# get_outliers_dynamic using each module's own stored thresholds -- the
# exact recomputation the simulation notebook's hooks performed. This cell
# only ENCODES those repo-produced values into packed storage, losslessly:
#
#   Keys  (per-channel, static nuq): each channel has at most 4 distinct
#     centroid values ever (2-bit). A per-layer [n_channels, 4] fp16
#     codebook is discovered lazily -- the first time a centroid value
#     appears in a channel it is assigned the next free 2-bit code; a 5th
#     distinct value would mean the data is not 2-bit and raises. Codes are
#     packed 4-per-byte into uint8.
#   Values (per-token, dynamic nuq): each token has at most 4 distinct
#     centroid values; a [4] fp16 codebook is stored PER TOKEN (dynamic
#     quantization genuinely must store per-token metadata -- this is real
#     storage cost, counted). Codes packed 4-per-byte into uint8.
#   Outliers (both): fp16 value + uint8 column index + int32 row index.
#
# Reconstruction (decode_*) is exact by construction -- codebook lookup
# returns the identical fp16 centroid, outliers are scattered back as the
# identical fp16 values. The validation cell asserts this bit-exactly.


def pack_codes_2bit(idx_u8):
    """[T, D] uint8 codes in 0..3 -> [T, D//4] uint8 (4 codes per byte)."""
    T, D = idx_u8.shape
    assert D % 4 == 0
    i = idx_u8.view(T, D // 4, 4)
    return (i[..., 0] | (i[..., 1] << 2) | (i[..., 2] << 4) | (i[..., 3] << 6)).contiguous()


def unpack_codes_2bit(packed_u8, D):
    """[T, D//4] uint8 -> [T, D] uint8 codes in 0..3."""
    T = packed_u8.shape[0]
    out = torch.empty(T, D // 4, 4, dtype=torch.uint8, device=packed_u8.device)
    out[..., 0] = packed_u8 & 0x3
    out[..., 1] = (packed_u8 >> 2) & 0x3
    out[..., 2] = (packed_u8 >> 4) & 0x3
    out[..., 3] = (packed_u8 >> 6) & 0x3
    return out.view(T, D)


def encode_perchannel_rows(y_half, outlier_mask, cb, cb_count):
    """Encode [T, D] fp16 rows against a growing per-channel codebook.
    cb: [D, 4] fp16 (persistent per layer), cb_count: [D] long.
    Returns idx [T, D] uint8 (code 0 at outlier positions -- overwritten by
    the outlier scatter at decode time)."""
    T, D = y_half.shape
    idx = torch.zeros(T, D, dtype=torch.uint8, device=y_half.device)
    slot_range = torch.arange(4, device=y_half.device)
    for t in range(T):
        row = y_half[t]
        m = outlier_mask[t]
        # Only match against codebook slots ALREADY assigned a real value
        # (slot_index < cb_count[channel]) -- cb starts zero-initialized,
        # so an unclaimed slot is indistinguishable from a legitimate
        # centroid value of exactly 0.0 without this guard. Without it, a
        # channel whose first-ever value happens to be 0.0 "claims" slot 0
        # for free (cb_count never increments for it), and a later
        # genuinely distinct value needing a new slot claims that SAME
        # slot 0 -- silently overwriting the real 0.0 centroid and
        # corrupting every earlier row that used it. nuq centroids very
        # plausibly include an exact 0.0 among thousands of channels,
        # which is what triggered "K reconstruction not bit-exact" in the
        # validation cell.
        claimed = slot_range.unsqueeze(0) < cb_count.unsqueeze(1)  # [D, 4]
        eq = (row.unsqueeze(1) == cb) & claimed  # [D, 4] exact fp16 match, claimed slots only
        has = eq.any(dim=1)
        pos = eq.float().argmax(dim=1)
        need_new = (~has) & (~m)
        if need_new.any():
            ch = torch.nonzero(need_new, as_tuple=False).squeeze(1)
            slots = cb_count[ch]
            if (slots >= 4).any():
                raise RuntimeError(
                    "Per-channel codebook overflow (5th distinct non-outlier value in a "
                    "channel) -- the incoming values are not 2-bit nuq output.")
            cb[ch, slots] = row[ch]
            cb_count[ch] = slots + 1
            pos[ch] = slots
        idx[t] = pos.to(torch.uint8)
        idx[t][m] = 0
    return idx


def decode_perchannel(idx_u8, cb):
    """idx [T, D] uint8 + cb [D, 4] fp16 -> [T, D] fp16."""
    D = idx_u8.shape[1]
    cols = torch.arange(D, device=idx_u8.device).unsqueeze(0).expand_as(idx_u8)
    return cb[cols, idx_u8.long()]


def encode_pertoken_rows(y_half, outlier_mask):
    """Encode [T, D] fp16 rows, one fresh 4-entry codebook per token.
    Returns (idx [T, D] uint8, cb [T, 4] fp16)."""
    T, D = y_half.shape
    idx = torch.zeros(T, D, dtype=torch.uint8, device=y_half.device)
    cb = torch.zeros(T, 4, dtype=torch.float16, device=y_half.device)
    for t in range(T):
        m = outlier_mask[t]
        uniq = torch.unique(y_half[t][~m])
        if uniq.numel() > 4:
            raise RuntimeError(
                "Per-token codebook overflow (>4 distinct non-outlier values in a token) "
                "-- the incoming values are not 2-bit nuq output.")
        cb[t, :uniq.numel()] = uniq
        eq = y_half[t].unsqueeze(1) == cb[t]   # [D, 4]
        idx[t] = eq.float().argmax(dim=1).to(torch.uint8)
        idx[t][m] = 0
    return idx, cb


def decode_pertoken(idx_u8, cb):
    """idx [T, D] uint8 + cb [T, 4] fp16 -> [T, D] fp16."""
    return torch.gather(cb, 1, idx_u8.long())


In [ ]:
# Block - REAL packed 2-bit KV storage: the per-layer store and session.
#
# PackedProjStore holds ONE projection's (K or V) packed cache for one
# layer: preallocated uint8 code buffer, per-token V codebooks (K's static
# per-layer codebook lives outside, since it persists across sequences),
# and flat outlier arrays. All buffers live on the GPU. Byte accounting is
# a running counter over what is physically appended, so reading the
# current cache size is O(1) and never touches tensor data (the same cheap
# shape-read discipline as the H2O notebooks).
#
# KVSession bundles the 22 layers x (K store, V store) for one sequence.
# reset_kv_session() starts a fresh sequence (fresh stores; K's static
# codebooks persist -- they are model metadata, discovered once, and their
# constant byte cost is counted separately in kv_bytes()).

N_LAYERS = model_q.config.num_hidden_layers
N_KV_HEADS = int(getattr(model_q.config, "num_key_value_heads", model_q.config.num_attention_heads))
N_Q_HEADS = model_q.config.num_attention_heads
HEAD_DIM = model_q.config.hidden_size // N_Q_HEADS
KV_DIM = N_KV_HEADS * HEAD_DIM
KV_GROUP = N_Q_HEADS // N_KV_HEADS
MAX_POS = C + GSM8K_MAX_NEW_TOKENS + 8

print(f"Layers: {N_LAYERS} | query heads: {N_Q_HEADS} | KV heads: {N_KV_HEADS} "
      f"| head_dim: {HEAD_DIM} | packed KV row: {KV_DIM} values -> {KV_DIM // 4} bytes")

# Static per-channel codebooks for K, one per layer, discovered lazily and
# persistent for the whole notebook run.
K_STATIC_CB = [torch.zeros(KV_DIM, 4, dtype=torch.float16, device=device) for _ in range(N_LAYERS)]
K_STATIC_CB_COUNT = [torch.zeros(KV_DIM, dtype=torch.long, device=device) for _ in range(N_LAYERS)]
K_STATIC_CB_BYTES = KV_DIM * 4 * 2  # per layer, constant


class PackedProjStore:
    OUTLIER_CAP_INIT = 4096

    def __init__(self, capacity):
        self.capacity = capacity
        self.n_tokens = 0
        self.codes = torch.zeros(capacity, KV_DIM // 4, dtype=torch.uint8, device=device)
        self.cb_pertoken = None  # allocated for V stores on first append
        self.out_cap = self.OUTLIER_CAP_INIT
        self.out_n = 0
        self.out_row = torch.zeros(self.out_cap, dtype=torch.int32, device=device)
        self.out_col = torch.zeros(self.out_cap, dtype=torch.uint8, device=device)
        self.out_val = torch.zeros(self.out_cap, dtype=torch.float16, device=device)
        self.byte_count = 0

    def _ensure_outlier_room(self, n_new):
        while self.out_n + n_new > self.out_cap:
            self.out_cap *= 2
            for name in ("out_row", "out_col", "out_val"):
                old = getattr(self, name)
                new = torch.zeros(self.out_cap, dtype=old.dtype, device=device)
                new[: self.out_n] = old[: self.out_n]
                setattr(self, name, new)

    def append_packed(self, idx_u8, outlier_mask, y_half, cb_rows=None):
        """idx_u8 [T, KV_DIM] codes; outlier_mask [T, KV_DIM] bool; y_half the
        fp16 rows (source of outlier values); cb_rows [T, 4] fp16 for V."""
        T = idx_u8.shape[0]
        assert self.n_tokens + T <= self.capacity, "packed store capacity exceeded"
        packed = pack_codes_2bit(idx_u8)
        self.codes[self.n_tokens: self.n_tokens + T] = packed
        self.byte_count += packed.numel()  # uint8 -> 1 byte per element
        if cb_rows is not None:
            if self.cb_pertoken is None:
                self.cb_pertoken = torch.zeros(self.capacity, 4, dtype=torch.float16, device=device)
            self.cb_pertoken[self.n_tokens: self.n_tokens + T] = cb_rows
            self.byte_count += cb_rows.numel() * 2
        orow, ocol = torch.nonzero(outlier_mask, as_tuple=True)
        n_new = orow.numel()
        if n_new > 0:
            self._ensure_outlier_room(n_new)
            self.out_row[self.out_n: self.out_n + n_new] = (orow + self.n_tokens).to(torch.int32)
            self.out_col[self.out_n: self.out_n + n_new] = ocol.to(torch.uint8)
            self.out_val[self.out_n: self.out_n + n_new] = y_half[orow, ocol]
            self.out_n += n_new
            self.byte_count += n_new * (4 + 1 + 2)  # int32 row + uint8 col + fp16 val
        self.n_tokens += T

    def decode_all(self, static_cb=None):
        """Reconstruct the full [n_tokens, KV_DIM] fp16 matrix (exact)."""
        idx = unpack_codes_2bit(self.codes[: self.n_tokens], KV_DIM)
        if static_cb is not None:
            dense = decode_perchannel(idx, static_cb)
        else:
            dense = decode_pertoken(idx, self.cb_pertoken[: self.n_tokens])
        if self.out_n > 0:
            dense[self.out_row[: self.out_n].long(), self.out_col[: self.out_n].long()] = \
                self.out_val[: self.out_n]
        return dense


class KVSession:
    def __init__(self, capacity):
        self.k = [PackedProjStore(capacity) for _ in range(N_LAYERS)]
        self.v = [PackedProjStore(capacity) for _ in range(N_LAYERS)]

    def seq_len(self):
        return self.k[0].n_tokens

    def kv_bytes(self):
        """REAL bytes physically resident for this sequence's packed cache:
        packed codes + per-token V codebooks + outlier (row+col+val) arrays,
        plus each layer's static K codebook (constant model-side metadata a
        real deployment must also hold). O(1) -- running counters only."""
        total = sum(s.byte_count for s in self.k) + sum(s.byte_count for s in self.v)
        total += N_LAYERS * K_STATIC_CB_BYTES
        return total


CURRENT_SESSION = None


def reset_kv_session(capacity=None):
    global CURRENT_SESSION
    CURRENT_SESSION = KVSession(capacity if capacity is not None else MAX_POS)
    return CURRENT_SESSION


DEBUG_EXACT_RECON = False  # validation cell flips this on for one short run


In [ ]:
# Block - REAL packed path: replace each layer's attention forward.
#
# Because the only persistent K/V is the packed store, attention must read
# from it: the stock LlamaAttention forward (which would cache fp16
# tensors) is replaced per layer with this implementation. Per call it:
#   1. runs the layer's own projections -- q_proj is stock; k_proj/v_proj
#      are the repo's QuantLinearSim, so their outputs are already the
#      method's quantized values (pre-RoPE for K, matching KVQuant's
#      per-channel-pre-RoPE design);
#   2. recomputes the raw pre-quantization projection (x @ weight, exactly
#      as the simulation notebook's tracking hooks did) and derives the
#      outlier mask with the repo's get_outliers/get_outliers_dynamic and
#      the module's own stored thresholds;
#   3. encodes + appends the new token(s) to the packed store -- after
#      this, the transient fp16 projections go out of scope;
#   4. reconstructs the FULL K (pre-RoPE) and V from the packed store,
#      applies RoPE to K at absolute positions 0..T-1 and to Q at its
#      absolute positions, expands KV heads to query heads (GQA), and runs
#      eager-math attention (fp32 softmax, matching stock eager).
#
# RoPE uses the model's own rotary_emb module, precomputed once for all
# positions 0..MAX_POS-1, so positions stay absolute and identical to the
# stock model's. Positions are derived from the store's own length before
# append -- the harness never needs to pass position_ids.
#
# Latency note (also in the intro): step 4 dequantizes the whole cache
# every call in unfused PyTorch ops. That is what dequantize-on-read
# genuinely costs in THIS implementation; deployment kernels fuse it.

import types

# Precompute cos/sin for all absolute positions using the model's own
# rotary embedding module (bit-identical frequencies to the stock model).
with torch.no_grad():
    _pos = torch.arange(MAX_POS, device=device).unsqueeze(0)
    _dummy = torch.zeros(1, 1, MAX_POS, HEAD_DIM, dtype=MODEL_DTYPE, device=device)
    _cos, _sin = model_q.model.rotary_emb(_dummy, _pos)
ROPE_COS = _cos.squeeze(0).to(MODEL_DTYPE)   # [MAX_POS, HEAD_DIM]
ROPE_SIN = _sin.squeeze(0).to(MODEL_DTYPE)
print("RoPE table:", tuple(ROPE_COS.shape), ROPE_COS.dtype)


def _rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def _apply_rope(x, positions):
    """x: [B, H, T, HEAD_DIM]; positions: [T] absolute."""
    cos = ROPE_COS[positions].unsqueeze(0).unsqueeze(0)
    sin = ROPE_SIN[positions].unsqueeze(0).unsqueeze(0)
    return (x * cos) + (_rotate_half(x) * sin)


def _true_projection(module, x2d):
    """Raw pre-quantization projection output, recomputed exactly as the
    simulation notebook's tracking hooks did: x @ weight (+ bias), half in,
    float out -- the tensor the repo's outlier functions expect."""
    y = x2d.half() @ module.weight.to(x2d.device)
    if module.bias is not None:
        y = y + module.bias.to(x2d.device)
    return y.float()


def _outlier_mask(module, y_true_2d):
    """The repo's own outlier identification, with the module's own stored
    thresholds -- identical logic to the simulation notebook's hooks."""
    if not module.include_sparse:
        return torch.zeros_like(y_true_2d, dtype=torch.bool)
    if module.dynamicquantization:
        mask = get_outliers_dynamic(
            y_true_2d, channel=module.ochannel, thresh=module.sparsity_threshold,
            first_few_fp16=module.first_few_fp16,
        )
    else:
        mask = get_outliers(
            y_true_2d, channel=module.ochannel,
            outlier_threshold_upper=module.outlier_threshold_upper,
            outlier_threshold_lower=module.outlier_threshold_lower,
            cap_outliers=module.cap_outliers,
            first_few_fp16=module.first_few_fp16,
        )
    return mask.reshape(y_true_2d.shape).bool()


def _packed_attention_forward(self, hidden_states, attention_mask=None,
                              position_ids=None, past_key_value=None,
                              output_attentions=False, use_cache=False,
                              cache_position=None, position_embeddings=None,
                              **kwargs):
    B, T, _ = hidden_states.shape
    assert B == 1, "packed path is single-sequence"
    li = self._packed_layer_idx
    sess = CURRENT_SESSION
    assert sess is not None, "call reset_kv_session() before running the model"
    past_len = sess.k[li].n_tokens

    x2d = hidden_states.reshape(-1, hidden_states.shape[-1])

    # 1. projections (k/v are the repo's QuantLinearSim -> quantized values)
    q = self.q_proj(hidden_states)
    y_k = self.k_proj(hidden_states).reshape(T, KV_DIM).half()
    y_v = self.v_proj(hidden_states).reshape(T, KV_DIM).half()

    # 2. repo-source outlier masks on the raw projections
    mask_k = _outlier_mask(self.k_proj, _true_projection(self.k_proj, x2d))
    mask_v = _outlier_mask(self.v_proj, _true_projection(self.v_proj, x2d))

    # 3. encode + append (packed storage is now the only persistent copy)
    idx_k = encode_perchannel_rows(y_k, mask_k, K_STATIC_CB[li], K_STATIC_CB_COUNT[li])
    sess.k[li].append_packed(idx_k, mask_k, y_k)
    idx_v, cb_v = encode_pertoken_rows(y_v, mask_v)
    sess.v[li].append_packed(idx_v, mask_v, y_v, cb_rows=cb_v)

    # 4. reconstruct full K/V from the packed store and attend
    k_pre = sess.k[li].decode_all(static_cb=K_STATIC_CB[li])      # [T_all, KV_DIM] fp16
    v_all = sess.v[li].decode_all()                                # [T_all, KV_DIM] fp16

    if DEBUG_EXACT_RECON:
        assert torch.equal(k_pre[past_len: past_len + T], y_k), "K reconstruction not bit-exact"
        assert torch.equal(v_all[past_len: past_len + T], y_v), "V reconstruction not bit-exact"

    T_all = k_pre.shape[0]
    k = k_pre.view(1, T_all, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    v = v_all.view(1, T_all, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    q = q.view(1, T, N_Q_HEADS, HEAD_DIM).transpose(1, 2)

    all_pos = torch.arange(T_all, device=device)
    k = _apply_rope(k, all_pos)
    q = _apply_rope(q, all_pos[past_len: past_len + T])

    if KV_GROUP > 1:
        k = k.repeat_interleave(KV_GROUP, dim=1)
        v = v.repeat_interleave(KV_GROUP, dim=1)

    scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(HEAD_DIM)
    if T > 1:
        qpos = all_pos[past_len: past_len + T].unsqueeze(1)   # [T, 1]
        kpos = all_pos.unsqueeze(0)                            # [1, T_all]
        causal = kpos > qpos                                   # True = masked
        scores = scores.masked_fill(causal.unsqueeze(0).unsqueeze(0), torch.finfo(scores.dtype).min)

    attn = torch.softmax(scores.float(), dim=-1).to(q.dtype)   # fp32 softmax, as stock eager
    out = torch.matmul(attn, v)
    out = out.transpose(1, 2).contiguous().view(1, T, N_Q_HEADS * HEAD_DIM)
    out = self.o_proj(out)
    return out, None, None


# Install: replace each layer's attention forward, keeping originals so the
# validation cell can temporarily restore the stock (simulation) path.
_ORIGINAL_ATTN_FORWARDS = {}
for _li, _layer in enumerate(model_q.model.layers):
    _attn = _layer.self_attn
    _attn._packed_layer_idx = _li
    _ORIGINAL_ATTN_FORWARDS[_li] = _attn.forward
    _attn.forward = types.MethodType(_packed_attention_forward, _attn)


def restore_stock_attention():
    for _li, _layer in enumerate(model_q.model.layers):
        _layer.self_attn.forward = _ORIGINAL_ATTN_FORWARDS[_li]


def install_packed_attention():
    for _li, _layer in enumerate(model_q.model.layers):
        _attn = _layer.self_attn
        _attn.forward = types.MethodType(_packed_attention_forward, _attn)


print(f"Packed-path attention installed on {len(_ORIGINAL_ATTN_FORWARDS)} layers.")


In [ ]:
# Block - VALIDATION: packed path vs. the simulation path, before any
# evaluation runs. Two checks on a short real prompt:
#
#   1. Bit-exact storage: with DEBUG_EXACT_RECON on, every attention call
#      asserts that decoding the packed store returns exactly the fp16
#      values the repo's QuantLinearSim produced. If encoding lost
#      anything, this raises immediately.
#   2. Logit agreement: the same tokens are run through (a) the stock
#      forward with QuantLinearSim -- i.e. the simulation notebook's exact
#      compute path -- and (b) the packed path. The quantized values are
#      identical by check 1, so remaining differences can only be fp16
#      summation-order effects between eager-math implementations; the
#      last-position logits must agree closely and share the same argmax.

_val_text = "The capital of France is Paris. The capital of Germany is"
_val_ids = tokenizer(_val_text, return_tensors="pt")["input_ids"].to(device)
print("Validation prompt tokens:", _val_ids.shape[1])

with torch.no_grad():
    restore_stock_attention()
    model_q.config.use_cache = False
    _ref_logits = model_q(input_ids=_val_ids, use_cache=False, return_dict=True).logits
    install_packed_attention()

    DEBUG_EXACT_RECON = True
    reset_kv_session()
    _real_logits = model_q(input_ids=_val_ids, return_dict=True).logits
    DEBUG_EXACT_RECON = False

_diff = (_ref_logits.float() - _real_logits.float()).abs()
_agree = (_ref_logits.argmax(-1) == _real_logits.argmax(-1)).float().mean().item()
print(f"Bit-exact packed-store reconstruction: PASSED (asserted inside every attention call)")
print(f"max |logit diff| vs simulation path: {_diff.max().item():.4f} | mean: {_diff.mean().item():.6f}")
print(f"argmax agreement across positions: {_agree:.3f}")
_bytes = CURRENT_SESSION.kv_bytes()
_dense = _val_ids.shape[1] * N_LAYERS * 2 * KV_DIM * 2
print(f"packed cache for this prompt: {_bytes} bytes (dense fp16 would be {_dense} bytes; "
      f"ratio {_bytes / _dense:.3f})")
assert _agree > 0.95, "packed path disagrees with the simulation path -- do not proceed"
print("VALIDATION PASSED")


## Helper Functions

Shared inference machinery used across all four datasets (WikiText-103,
GSM8K, ARC-Challenge, HellaSwag).


In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook for timing-safe GPU synchronization and
# between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()


In [ ]:
# Block - Packed-path stepping + memory readout -- shared inference
# machinery for every dataset.
#
#   real_reset(capacity) -- fresh packed session for a new sequence.
#   real_step(token_tensor) -- one forward call through the packed path;
#       positions come from the store's own length, so callers just feed
#       tokens in order. The quantize + pack + dequantize-on-read work all
#       happens INSIDE the model call, so it sits INSIDE the callers' timed
#       brackets -- deliberately: it is this implementation's real
#       per-token cost, exactly as eviction sits inside H2O's timed steps.
#   real_prefill(input_ids) -- one batched forward over a whole prompt
#       (multi-token; the packed attention handles the causal mask).
#
# Memory: session_kv_bytes() reads the running byte counters -- O(1), no
# tensor traffic, safe to call anywhere. The packed cache only ever GROWS
# within a sequence (no eviction), so the value at the end of a sequence
# IS the peak; no separate untimed measurement pass is needed, and there
# is no tracker machinery -- the number is the real resident byte count,
# the same kind of MEASURED figure the H2O notebooks report.


def real_reset(capacity=None):
    return reset_kv_session(capacity)


@torch.no_grad()
def real_step(token_tensor):
    outputs = model_q(input_ids=token_tensor, return_dict=True)
    return outputs.logits[:, -1, :]


@torch.no_grad()
def real_prefill(input_ids):
    outputs = model_q(input_ids=input_ids, return_dict=True)
    return outputs.logits


def session_kv_bytes():
    return CURRENT_SESSION.kv_bytes() if CURRENT_SESSION is not None else 0


In [ ]:
# Block - Shared multiple-choice (MC) scoring machinery, used by BOTH
# ARC-Challenge and HellaSwag -- same structure and metric definitions as
# the rest of the notebook family; only the per-step mechanics go through
# the packed path.
#
# lm_eval_encode_pair -- identical to the family (joint tokenization then
# split at the context's own token count, matching lm-evaluation-harness;
# left-truncates the context if the pair would exceed C).
#
# score_mc_choice_real -- times the full step-by-step walk over one
# choice's tokens through real_step (packed quantize/pack/decode inside
# the timed bracket, as the method's real per-token cost). Memory is the
# session's real byte count at the end of the walk (the packed cache only
# grows, so end == peak).
#
# score_mc_question_real -- same aggregation as the family: raw/normalized
# accuracy, perplexity from the gold choice only, TTFT = mean across
# choices, TBT = weighted mean across every choice's decode steps, total
# latency = SUM across choices, peak memory = MAX across choices.


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")
    if len(continuation_ids) > int(C):
        raise ValueError(f"Choice has {len(continuation_ids)} tokens, exceeding C={C}.")

    max_context_tokens = int(C) + 1 - len(continuation_ids)
    if max_context_tokens < 1:
        raise ValueError("No room remains for a context token.")
    context_ids = context_ids[-max_context_tokens:]

    return context_ids, continuation_ids


@torch.no_grad()
def score_mc_choice_real(prompt, choice):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    full_ids_1d = torch.tensor(context_ids + continuation_ids, device=device)
    n_context = len(context_ids)

    chunk_ids = full_ids_1d.unsqueeze(0)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    real_reset(full_ids_1d.shape[0] + 4)

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        step_logits = real_step(step_input)
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        step_target = target_ids[:, pos]

        if pos + 1 >= n_context:
            loss = loss_fct(step_logits, step_target)
            nll_sum += loss.float().item()
            scored += 1

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    peak_bytes = session_kv_bytes()   # packed cache only grows -> end == peak

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_real(prompt, choices, gold_index):
    choice_results = [score_mc_choice_real(prompt, choice) for choice in choices]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }


## WikiText-103

In [ ]:
# Block - WikiText-103: load the FULL test text, concatenate ALL of
# it into one long sequence, tokenize, split into consecutive chunks of
# length C, then cap to the FIRST N_EVAL_SAMPLES chunks -- a deterministic
# prefix of the real test set, not a random sample. VERBATIM from the
# family so all methods are evaluated on token-for-token identical data.
# The download goes through robust_call so a transient network blip gets
# retried instead of killing the whole run, clearing its HF cache between
# retries (on_retry), since a broken 157MB download can leave a corrupted
# partial file that plain retries just resume (and re-break at the same
# point) forever.


def chunk_sequence(token_ids_1d, chunk_len):
    chunks = []
    n = token_ids_1d.shape[0]
    for start in range(0, n, chunk_len):
        chunks.append(token_ids_1d[start:start + chunk_len])
    return chunks


def load_wikitext103_chunks():
    testdata = robust_call(
        load_dataset, "wikitext", "wikitext-103-raw-v1", split="test",
        desc="WikiText-103 test load", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
    )
    texts = [t for t in testdata["text"] if len(t.strip()) > 0]
    full_text = "\n\n".join(texts)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    print(f"WikiText-103 test set: {len(texts)} non-empty lines, {ids.shape[0]} tokens total")
    return chunk_sequence(ids, C)[:N_EVAL_SAMPLES]


wikitext103_chunks = load_wikitext103_chunks()
print(f"WikiText-103: {len(wikitext103_chunks)} chunks of up to {C} tokens (first N_EVAL_SAMPLES={N_EVAL_SAMPLES}, not randomly selected)")


In [ ]:
# Block - WikiText-103 step-by-step eval function. TTFT/TBT/perplexity/
# accuracy come ONLY from the timed step-by-step loop below -- the model
# is stepped one token at a time through the packed path, teacher-forced
# (real corpus tokens fed in, never the model's own prediction). The
# packed quantize/pack/decode work happens INSIDE the timed forward call
# -- it is this implementation's real per-token cost. Memory is the
# session's real byte count read AFTER the loop (the packed cache only
# grows within a sequence, so the final value IS the peak); the read is
# O(1) counter access and touches no tensors.


@torch.no_grad()
def score_chunk_real(chunk_ids_1d):
    L = chunk_ids_1d.shape[0]
    if L < 2:
        return None
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    real_reset(L + 4)

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    correct = 0
    scored = 0
    step_times = []

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        step_logits = real_step(step_input)
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        step_target = target_ids[:, pos]

        loss = loss_fct(step_logits, step_target)
        nll_sum += loss.float().item()

        pred = step_logits.argmax(dim=-1)
        correct += int((pred == step_target).sum().item())
        scored += 1

    ttft_sec = step_times[0]
    tbt_sec = sum(step_times[1:]) / len(step_times[1:]) if len(step_times) > 1 else 0.0
    total_latency_sec = sum(step_times)

    peak_bytes = session_kv_bytes()

    return {
        "nll_sum": nll_sum, "scored": scored, "correct": correct,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def preview_chunk_prediction(chunk_ids_1d, n_preview_tokens=30):
    """Untimed, separate pass over just the first n_preview_tokens of a
    chunk -- for printing what the model actually predicted vs. the real
    next tokens. Runs through the same packed path (own fresh session) so
    the preview reflects the method's behavior. Does not affect any
    reported metric."""
    n_preview_tokens = min(n_preview_tokens, chunk_ids_1d.shape[0] - 1)
    if n_preview_tokens < 1:
        return None
    preview_ids = chunk_ids_1d[:n_preview_tokens + 1].unsqueeze(0).to(device)
    input_ids = preview_ids[:, :-1]
    target_ids = preview_ids[:, 1:]

    real_reset(n_preview_tokens + 4)
    preds = []
    for pos in range(input_ids.shape[1]):
        step_logits = real_step(input_ids[:, pos:pos + 1])
        preds.append(int(step_logits.argmax(dim=-1)[0].item()))

    return {
        "prompt_text": tokenizer.decode(input_ids[0], skip_special_tokens=True),
        "actual_next_text": tokenizer.decode(target_ids[0], skip_special_tokens=True),
        "predicted_next_text": tokenizer.decode(preds, skip_special_tokens=True),
    }


In [ ]:
# Block - WikiText-103 driver: run every chunk through score_chunk_real.
# Aggregation is identical to the family: perplexity = exp(pooled mean NLL
# over all scored tokens), accuracy = pooled next-token accuracy, TTFT/TBT/
# latency = means over chunks, peak_memory_gb = max over chunks,
# average_memory_gb = mean over chunks. Same output columns.


def evaluate_lm_dataset_real(dataset_name, chunks, method_label):
    clear_memory()
    total_nll = 0.0
    total_scored = 0
    total_correct = 0
    ttft_values, tbt_values, latency_values, peak_mem_values = [], [], [], []

    N_PREVIEW_CHUNKS = 5
    for chunk_idx, chunk_ids in enumerate(tqdm(chunks, desc=f"{dataset_name} | {method_label}")):
        if chunk_idx < N_PREVIEW_CHUNKS:
            preview = preview_chunk_prediction(chunk_ids)
            if preview is not None:
                print(f"\n--- {dataset_name} | {method_label} | chunk {chunk_idx} preview ---")
                print(f"Prompt:              {preview['prompt_text']!r}")
                print(f"Actual next text:    {preview['actual_next_text']!r}")
                print(f"Predicted next text: {preview['predicted_next_text']!r}")

        result = score_chunk_real(chunk_ids)
        if result is None:
            continue
        total_nll += result["nll_sum"]
        total_scored += result["scored"]
        total_correct += result["correct"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

    avg_nll = total_nll / max(total_scored, 1)
    perplexity = math.exp(min(avg_nll, 50.0))
    accuracy = total_correct / max(total_scored, 1)

    return {
        "dataset": dataset_name,
        "method": method_label,
        "perplexity": perplexity,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


lm_results = []
for _name, _chunks in [("WikiText-103", wikitext103_chunks)]:
    lm_results.append(evaluate_lm_dataset_real(_name, _chunks, METHOD_NAME))

lm_results_df = pd.DataFrame(lm_results)
display(lm_results_df)


## GSM8K

In [ ]:
# Block - GSM8K loading: canonical source, sequential first N_QA_SAMPLES
# questions (no shuffling), few-shot prompt -- VERBATIM from the family so
# all methods see the exact same question set and prompts. Download goes
# through robust_call, clearing its HF cache between retries.


def extract_gsm8k_gold_answer(answer_text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None


gsm8k_test = robust_call(
    load_dataset, "gsm8k", "main", split="test",
    desc="GSM8K test load", on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_qa_pairs = []
for i in range(min(N_QA_SAMPLES, len(gsm8k_test))):
    item = gsm8k_test[i]
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        gsm8k_qa_pairs.append({"question": item["question"], "gold": gold, "gold_text": item["answer"]})

print(f"GSM8K: {len(gsm8k_qa_pairs)} questions loaded (of {len(gsm8k_test)} total in test set)")


In [ ]:
# Block - GSM8K generation through the packed path. model.generate() with
# use_cache would build its own fp16 cache and never touch the packed
# store, and with use_cache=False it would re-feed the full sequence each
# step (re-appending duplicates to the store) -- so generation is a manual
# prefill + greedy decode loop, exactly the structure the H2O notebooks
# use (their engines can't ride inside generate() either), with the same
# timing definitions as the whole family:
#   TTFT  = time from generation start until the first generated token's
#           logits are ready (the batched prefill forward).
#   total = wall time of the whole generation.
#   TBT   = (total - TTFT) / max(n_generated - 1, 1).
# n_generated counts every token an actual forward call produced,
# INCLUDING the final EOS-producing step (matching the family's
# StoppingCriteria, which fires after that step too) -- tracked as
# n_forward_tokens, while gen_text excludes EOS itself.
#
# Answer grading is VERBATIM from the family: truncate the generation at
# the first "Question:", extract "#### <num>" (falling back to the last
# number), compare |pred - gold| < 1e-4.
#
# Memory: the session's real packed byte count at the end (the store only
# grows, so end == peak). No separate measurement pass exists or is
# needed -- the number is MEASURED, like H2O's.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


@torch.no_grad()
def generate_gsm8k_real(question):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    real_reset(prompt_len + GSM8K_MAX_NEW_TOKENS + 4)

    sync_if_cuda()
    gen_start = time.perf_counter()

    # ---- Prefill: one batched forward through the packed path ----
    logits = real_prefill(enc["input_ids"])
    next_id = int(logits[:, -1, :].argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    generated_ids = []
    n_forward_tokens = 1  # the prefill forward already produced next_id

    # ---- Greedy decode ----
    for step in range(GSM8K_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        if step == GSM8K_MAX_NEW_TOKENS - 1:
            break  # no wasted forward for a token we would never use

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        last_logits = real_step(token_tensor)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated
    peak_bytes = session_kv_bytes()

    return {
        "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def score_gsm8k_perplexity_real(question, gold_text):
    """Secondary, untimed teacher-forced pass on the gold answer -- same
    definition as the family (perplexity of the gold final number given the
    prompt), run through the packed path: one batched forward over
    prompt+gold, reading each gold token's log-probability at its
    position."""
    gold_number = extract_gsm8k_gold_answer(gold_text)
    if gold_number is None:
        return None
    gold_str = str(int(gold_number)) if gold_number == int(gold_number) else str(gold_number)
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    target_ids = tokenizer(" " + gold_str, add_special_tokens=False)["input_ids"]
    full_ids = torch.tensor([prompt_ids + target_ids], device=device)

    real_reset(full_ids.shape[1] + 4)
    logits = real_prefill(full_ids)[0]
    n_prompt = len(prompt_ids)
    nll_sum = 0.0
    scored = 0
    for ti, target_id in enumerate(target_ids):
        pos = n_prompt - 1 + ti
        if pos < 0 or pos >= logits.shape[0]:
            continue
        log_probs = torch.log_softmax(logits[pos].float(), dim=-1)
        nll_sum += -log_probs[target_id].item()
        scored += 1
    if scored == 0:
        return None
    return math.exp(min(nll_sum / scored, 50.0))


In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_real
# (accuracy + TTFT/TBT/latency + memory, all from the SAME real generation
# call), plus the secondary teacher-forced perplexity pass. Aggregation is
# identical to the family: accuracy = fraction correct, perplexity = MEAN
# of per-question perplexities, TTFT/TBT/latency = means, peak_memory_gb =
# max over questions, average_memory_gb = mean over questions. Same output
# columns.


def evaluate_gsm8k_real(qa_pairs, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_real(qa["question"])
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = score_gsm8k_perplexity_real(qa["question"], qa["gold_text"])
        if ppl is not None:
            ppl_values.append(ppl)

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_real(gsm8k_qa_pairs, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)


## ARC-Challenge

Likelihood-based multiple-choice scoring (no generation): every answer
choice is scored (raw and character-length-normalized), perplexity from
the gold choice, TTFT/TBT/latency/memory aggregated across all choices.


In [ ]:
# Block - ARC-Challenge loading: canonical source (allenai/ai2_arc,
# ARC-Challenge config), sequential first N_QA_SAMPLES questions (no
# shuffling), matching the GSM8K loading convention exactly -- first
# min(N_QA_SAMPLES, len(testdata)) items by index, skipping only the rare
# item whose answerKey doesn't match any of its own choice labels (a
# data-quality edge case, not a sampling choice). Download goes through
# robust_call, clearing its HF cache between retries.


def load_arc_challenge_items():
    testdata = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    items = []
    for i in range(min(N_QA_SAMPLES, len(testdata))):
        row = testdata[i]
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })
    return items


arc_items = load_arc_challenge_items()
print(f"ARC-Challenge: {len(arc_items)} questions loaded (first N_QA_SAMPLES={N_QA_SAMPLES}, not randomly selected)")


In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_real (Helper Functions), then
# reports BOTH standard MC accuracy metrics (raw + character-length
# normalized). Aggregation mirrors GSM8K: perplexity = MEAN of
# per-question perplexities (from each question's gold choice), TTFT/TBT/
# latency = means over questions, peak_memory_gb = max over questions,
# average_memory_gb = mean over questions.


def evaluate_arc_real(items, method_label):
    clear_memory()
    raw_correct = 0
    normalized_correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_real(prompt, choice_texts, gold_index)

        raw_correct += result["raw_correct"]
        normalized_correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            predicted_label = item["choices"][result["normalized_prediction"]][0]
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

    accuracy_raw = raw_correct / max(total, 1)
    accuracy_normalized = normalized_correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy_normalized,
        "accuracy_raw": accuracy_raw,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


arc_results = [
    evaluate_arc_real(arc_items, METHOD_NAME),
]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)


## HellaSwag

Likelihood-based multiple-choice scoring (no generation), same
methodology as ARC-Challenge: every answer choice is scored (raw and
character-length-normalized), perplexity from the gold ending,
TTFT/TBT/latency/memory aggregated across all choices.


In [ ]:
# Block - HellaSwag loading: canonical source (Rowan/hellaswag), sequential
# first N_QA_SAMPLES examples (no shuffling) from the validation split --
# HellaSwag's test split has no public labels, so validation is the
# standard evaluation split (matching lm-evaluation-harness convention).
# Preprocessing matches the standard lm-evaluation-harness HellaSwag
# convention: strip "[title]" markers and bracketed annotations, collapse
# double spaces. Download goes through robust_call, clearing its HF cache
# between retries.


def _hellaswag_preprocess(text):
    """Matches lm-evaluation-harness HellaSwag preprocessing."""
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def load_hellaswag_items():
    dataset = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    items = []
    for i in range(min(N_QA_SAMPLES, len(dataset))):
        row = dataset[i]
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = _hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [_hellaswag_preprocess(choice) for choice in row["endings"]]
        items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })
    return items


hellaswag_items = load_hellaswag_items()
print(f"HellaSwag: {len(hellaswag_items)} validation examples loaded (first N_QA_SAMPLES={N_QA_SAMPLES}, not randomly selected)")


In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_real (Helper Functions) -- the same
# machinery ARC-Challenge uses. Aggregation is identical to ARC-Challenge:
# raw + normalized accuracy, perplexity = MEAN of per-question
# perplexities (from each example's gold ending), TTFT/TBT/latency =
# means over examples, peak_memory_gb = max over examples,
# average_memory_gb = mean over examples.


def evaluate_hellaswag_real(items, method_label):
    clear_memory()
    raw_correct = 0
    normalized_correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_real(item["prompt"], item["choices"], item["gold_index"])

        raw_correct += result["raw_correct"]
        normalized_correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

    accuracy_raw = raw_correct / max(total, 1)
    accuracy_normalized = normalized_correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy_normalized,
        "accuracy_raw": accuracy_raw,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


hellaswag_results = [
    evaluate_hellaswag_real(hellaswag_items, METHOD_NAME),
]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)


## Save Results

In [ ]:
# Block - Combine WikiText-103 / GSM8K / ARC-Challenge / HellaSwag results
# into one table with ONLY the specified metrics, then save to CSV --
# identical column schema to the rest of the family, saved into the same
# Drive folder, so all methods' CSVs concatenate directly into one
# comparison table.
#
# Cross-method comparison reminders:
#   - This notebook's memory numbers are MEASURED (real packed tensors) --
#     directly comparable to the H2O notebooks' measured numbers, unlike
#     the simulation notebook's calculated ones.
#   - Latency here is real but UNOPTIMIZED (unfused dequantize-on-read +
#     eager-math attention). Compare only against an eager-attention
#     baseline run, and never present it as deployment-kernel KVQuant
#     speed.
#   - Accuracy/perplexity should closely match the simulation notebook's
#     kvquant_2bit rows -- if they don't, something is wrong; investigate
#     before trusting either.
#   - Confirm the GPU name printed in Block 2 matches every other
#     notebook's run before trusting any timing/memory comparison.
#
# Robust to partial runs: only concatenates whichever of
# lm_results_df / gsm8k_results_df / arc_results_df / hellaswag_results_df
# actually exist in this session. NOTE: "accuracy_raw" exists only in the
# MC datasets' rows; reindex(columns=...) fills it with NaN for
# WikiText/GSM8K rows even when no MC dataset was run, instead of raising.

_result_df_names = ["lm_results_df", "gsm8k_results_df", "arc_results_df", "hellaswag_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

results_df = pd.concat(_available_dfs, ignore_index=True)
results_df = results_df.reindex(columns=[
    "dataset", "method", "perplexity", "accuracy", "accuracy_raw",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_gb", "average_memory_gb",
])
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = "/content/drive/MyDrive/KVQuant_v3_Results/kvquant_abits2_real_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")
